# Relay ext 测试

In [1]:
import set_env

In [2]:
import numpy as np
import tvm
from tvm import relay

In [3]:
@tvm.register_func("relay.ext.test1")
def relay_ext_test(func):
    return None

mod = tvm.IRModule()
shape = (relay.Any(), 25)
dtype = "float32"

# external function
x = relay.var("x", shape=shape, dtype=dtype)
weight = relay.const(np.random.rand(5, 25).astype("float32"), dtype="float32")
out = relay.nn.dense(x, weight)
f1 = relay.Function([x], out)
f1 = f1.with_attr("Primitive", tvm.tir.IntImm("int32", 1))
f1 = f1.with_attr("Inline", tvm.tir.IntImm("int32", 1))
f1 = f1.with_attr("Compiler", "test1")
f1 = f1.with_attr("global_symbol", "f1")
glb_f1 = relay.GlobalVar("f1")
mod[glb_f1] = f1
mod = relay.transform.InferType()(mod)

# Main function
x = relay.var("x", shape=shape, dtype=dtype)
mod["main"] = relay.Function([x], glb_f1(x))
comp = relay.vm.VMCompiler()
opt_mod, _ = comp.optimize(mod, target="llvm")
assert "shape_func" in opt_mod.astext(False)

In [4]:
mod.show()

In [5]:
from tvm.relay.backend.vm import VMCompiler
from tvm.relay.dataflow_pattern import is_op, wildcard
from tvm.relay.loops import while_loop
from tvm.relay.prelude import Prelude
from tvm.relay.scope_builder import ScopeBuilder
@tvm.register_func("relay.ext.test2")
def relay_ext_test(func):
    return None

data_shape = (relay.Any(), 16)
weight_shape = (relay.Any(), 16)

dense = relay.nn.dense(
    relay.var("data", shape=data_shape), relay.var("weight", shape=weight_shape)
)
mod = tvm.IRModule.from_expr(dense)

patterns = [("test.dense", is_op("nn.dense")(wildcard(), wildcard()))]
passes = tvm.transform.Sequential(
    [
        relay.transform.MergeComposite(patterns),
        relay.transform.AnnotateTarget(["test2"]),
        relay.transform.PartitionGraph(),
    ]
)

mod = passes(mod)

compiler = VMCompiler()
compiler.lower(mod, "llvm")

Cannot find config for target=llvm -keys=cpu -mtriple=x86_64-unknown-linux-gnu, workload=('dense_pack.x86', ('TENSOR', (any_dim, 16), 'float32'), ('TENSOR', (any_dim, 16), 'float32'), None, 'float32'). A fallback configuration is used, which may bring great performance regression.


In [6]:
mod.show()